# Reference Counting in CPython

**Week 1 · Cycle 1 — Python internals: object model, memory, refcounting, GC**

Two rules explain almost everything in this notebook:

1. **Everything is an object.** A number, a list, a function, a class, a module — all of them live on the heap with a type, an id (address) and a reference count.
2. **A variable is not the object.** It is a *name* pointing at the object. Assignment never copies the object, it only adds another arrow.

CPython frees an object the moment nothing points at it any more — its reference count drops to `0`. The cycle collector (`gc`) exists only to clean up the case reference counting cannot handle: objects that point at each other.

```
   name              object on the heap
   ----              ------------------
    a  ──────────►  ┌───────────────────┐
                    │ type:   list      │
    b  ──────────►  │ value:  [1, 2, 3] │
                    │ refcount: 2       │
                    └───────────────────┘
```

## 0. Setup

In [24]:
import sys
import ctypes
import gc

## 1. Every object has a type, an id and a refcount

`id(obj)` is the object's address in memory (in CPython). Two names with the same `id` are **the same object**, not two equal copies.

In [25]:
a = [1, 2, 3]

print("type :", type(a))
print("id :", id(a))
print("id (hex):", hex(id(a)))

type : <class 'list'>
id : 4379810048
id (hex): 0x1050e9900


## 2. Counting references with `sys.getrefcount`

`sys.getrefcount(obj)` always reports **one extra** reference — the temporary argument passed into the function itself.

```
    a ──►  [1,2,3]          sys.getrefcount(a) -> 2
                                        (a) + (the argument)
```

In [26]:
a = [1, 2, 3]
print("refs after 'a = [1,2,3]' :", sys.getrefcount(a))   # 1 real + 1 argument

refs after 'a = [1,2,3]' : 2


### Assignment adds an arrow, it does not copy

```
  before:  a ──►  [1,2,3]                   count = 1

  b = a

  after:   a ──►  [1,2,3]  ◄── b            count = 2
```

In [27]:
b = a
print("b is a :", b is a)          # same object, not a copy
print("refs now :", sys.getrefcount(a))

b is a : True
refs now : 3


Removing one name only removes one arrow. The object survives as long as any name is left.

In [28]:
a = None                                # the name 'a' stops pointing at the list
print("refs after 'a = None':", sys.getrefcount(b))

refs after 'a = None': 2


## 3. Reading the true refcount with `ctypes`

The first field of every CPython object header **is** the reference count. If we read it straight from the address, nothing is added on top — so we see the real number, not `n + 1`.

```
  PyObject header at id(x)
  ┌──────────────┬──────────────┬─────────────┐
  │ ob_refcnt    │ ob_type      │  payload…   │
  │  ◄── this    │              │             │
  └──────────────┴──────────────┴─────────────┘
```

Note we pass the **address**, never the object — passing the object would add a reference of its own.

In [30]:
def refcount_at(address):
    """True refcount of the object living at `address` (CPython only)."""
    return ctypes.c_long.from_address(address).value

x = [1, 2]
x_id = id(x)

print("ctypes :", refcount_at(x_id))      # 1  -> the real count
print("sys :", sys.getrefcount(x))     # 2  -> real count + argument

ctypes : 1
sys : 2


In [31]:
y = x
print("after 'y = x' ->", refcount_at(x_id))

after 'y = x' -> 2


> ⚠️ `refcount_at` reads raw memory. Once the count hits `0` the object is gone and that address means nothing —
> reading it afterwards gives garbage, not a clean `0`.
> (Also: in Python 3.12+ objects like `None`, `True` and small ints are *immortal* and report a huge fixed refcount.)

## 4. Orphaned objects

An object nobody points at is an **orphan**. Its count reaches `0` and CPython destroys it *immediately* — no waiting for a collector.

```
   a ──►  Ghost(refcount=1)

   a = None      ─────────────►   Ghost(refcount=0)  ✗ destroyed right here
```

In [32]:
class Ghost:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"   [{self.name} destroyed]")

a = Ghost("ghost-1")
print("still alive...")
a = None                 # last reference dropped -> destroyed on this line
print("done")

still alive...
   [ghost-1 destroyed]
done


With two names, the object is only destroyed when the **second** one lets go.

In [33]:
a = Ghost("ghost-2")
b = a
print("refs :", sys.getrefcount(a) - 1)

del a
print("after 'del a' — still alive, b holds it")

del b
print("after 'del b' — nothing left")

refs : 2
after 'del a' — still alive, b holds it
   [ghost-2 destroyed]
after 'del b' — nothing left


## 5. Circular references — where refcounting fails

If two objects point at each other, deleting the outside names is not enough: each one is still kept alive **by the other**.

```
   a ──►  A ──────────►  B  ◄── b          A.ref = b,  B.ref = a
          ▲              │
          └──────────────┘

   del a, b:

          A ──────────►  B                 both counts are still 1
          ▲              │                 → unreachable, but never freed
          └──────────────┘                 → this is a leak
```

Below the cycle collector is switched off, so the leak is visible.

In [34]:
class Node:
    def __init__(self, name):
        self.name = name
        self.ref = None
    def __del__(self):
        print(f"   [{self.name} collected]")

gc.disable()             # turn the cycle collector off, so we can see the leak

a = Node("A")
b = Node("B")
a.ref = b                # A ──► B
b.ref = a                # B ──► A

a_id, b_id = id(a), id(b)
print("A refs:", refcount_at(a_id), " B refs:", refcount_at(b_id))   # 2 each: name + the other node

del a, b
print("after 'del a, b' -> nothing was collected")
print("A refs:", refcount_at(a_id), " B refs:", refcount_at(b_id))   # 1 each: only each other

A refs: 2  B refs: 2
after 'del a, b' -> nothing was collected
A refs: 1  B refs: 1


## 6. The garbage collector cleans the cycle

The `gc` module walks the heap, finds groups of objects that only reference each other and are unreachable from anywhere else, and frees the whole group.

```
   refcounting   →  frees everything that drops to 0        (immediate)
   gc            →  frees unreachable cycles                (periodic / on demand)
```

In [35]:
freed = gc.collect()      # run the cycle collector by hand
print("objects collected:", freed)

gc.enable()               # back to normal — cycles are handled automatically
print("gc enabled:", gc.isenabled())

   [A collected]
   [B collected]
objects collected: 254
gc enabled: True


## Takeaways

| Idea | What actually happens |
|---|---|
| Everything is an object | type + id + refcount live in the object header |
| `b = a` | a second arrow to one object, never a copy |
| `sys.getrefcount(x)` | true count **+ 1** (the argument) |
| `ctypes.c_long.from_address(id(x))` | the true count, read from the header |
| refcount hits 0 | destroyed instantly, `__del__` runs |
| cycle `A → B → A` | refcounting alone leaks it; `gc` frees it |

**Next in Week 1:** interning, copy vs reference (shallow / deep), and measured memory footprint of `list` / `tuple` / `array` / NumPy for the same data.